# Lunyoro-Rutooro Translator — Colab Backend

Runs NLLB-200 FP32 inference on a free Colab T4 GPU.
Models load from HuggingFace Hub. Tunnel via cloudflared. Auto-registers with Cloudflare Worker.

## Architecture
```
Browser → Vercel (horizonx.kathay.tech)
    └─ Cloudflare Worker (stable URL, never changes)
         └─ Colab T4 (this notebook) ← NLLB from HF Hub
              └─ cloudflared tunnel (auto *.trycloudflare.com)
```

## Steps
1. Runtime → Change runtime type → **T4 GPU**
2. Fill `HF_TOKEN` and `WORKER_SECRET` in **Cell 1**
3. **Runtime → Run all** — done in ~3 min


In [ ]:
# ── Cell 1 — SECRETS (fill these in before running) ───────────────────────────
HF_TOKEN      = ""                  # huggingface.co/settings/tokens
WORKER_SECRET = "lunyoro-colab-2026" # matches WORKER_SECRET set on the CF Worker

NLLB_EN2LUN_REPO = "keithtwesigye/lunyoro-nllb-en2lun"
NLLB_LUN2EN_REPO = "keithtwesigye/lunyoro-nllb-lun2en"
HF_CHAT_MODEL    = "meta-llama/Llama-3.1-8B-Instruct"
WORKER_URL        = "https://lunyoro-translator-api.keithtwesigye74.workers.dev"

assert HF_TOKEN, "❌ Paste your HF_TOKEN above first!"
print("✓ Secrets loaded")

In [ ]:
# ── Cell 2 — INSTALL DEPENDENCIES ─────────────────────────────────────────────
!pip install -q \
    fastapi "uvicorn[standard]" \
    "transformers==4.51.3" sentencepiece accelerate \
    easyocr opencv-python-headless pillow \
    openai rapidfuzz python-multipart \
    PyPDF2 python-docx pdfplumber
print("✓ Dependencies installed")

In [ ]:
# ── Cell 3 — CHECK GPU ────────────────────────────────────────────────────────
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠ No GPU — go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# ── Cell 4 — LOAD NLLB MODELS ─────────────────────────────────────────────────
import os
os.environ["HF_TOKEN"]      = HF_TOKEN
os.environ["HF_CHAT_MODEL"] = HF_CHAT_MODEL

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Loading NLLB en→lun …")
tok_en2lun = AutoTokenizer.from_pretrained(NLLB_EN2LUN_REPO, src_lang="eng_Latn", token=HF_TOKEN)
mdl_en2lun = AutoModelForSeq2SeqLM.from_pretrained(NLLB_EN2LUN_REPO, token=HF_TOKEN).to(device)
mdl_en2lun.eval()
print("  ✓ en→lun ready")

print("Loading NLLB lun→en …")
tok_lun2en = AutoTokenizer.from_pretrained(NLLB_LUN2EN_REPO, src_lang="run_Latn", token=HF_TOKEN)
mdl_lun2en = AutoModelForSeq2SeqLM.from_pretrained(NLLB_LUN2EN_REPO, token=HF_TOKEN).to(device)
mdl_lun2en.eval()
print("  ✓ lun→en ready")

NLLB_LANG_LUN = "run_Latn"
NLLB_LANG_EN  = "eng_Latn"
TGT_LUN_ID = tok_en2lun.convert_tokens_to_ids(NLLB_LANG_LUN)
TGT_EN_ID  = tok_lun2en.convert_tokens_to_ids(NLLB_LANG_EN)
print(f"  Token IDs — run_Latn:{TGT_LUN_ID}  eng_Latn:{TGT_EN_ID}")

In [ ]:
# ── Cell 5 — HELPERS + MOBILENET + EASYOCR ────────────────────────────────────
import re, io

def _sentence_case(text: str) -> str:
    text = text.strip()
    if text and not text[0].isupper():
        text = text[0].upper() + text[1:]
    if text and text[-1] not in ".!?,;:":
        qwords = ("how","what","where","when","who","why","which",
                  "is ","are ","do ","does ","can ","will ","have ")
        text += "?" if any(text.lower().startswith(w) for w in qwords) else "."
    return text

def translate_en2lun(text: str) -> str:
    text = _sentence_case(text)
    tok_en2lun.src_lang = NLLB_LANG_EN
    inputs = tok_en2lun(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        out = mdl_en2lun.generate(**inputs, forced_bos_token_id=TGT_LUN_ID,
                                   num_beams=5, max_length=256,
                                   no_repeat_ngram_size=3, repetition_penalty=1.3)
    result = tok_en2lun.decode(out[0], skip_special_tokens=True)
    return re.sub(r"^(run_Latn|eng_Latn|nyk_Latn)\s*:\s*", "", result.replace("▁", " ")).strip()

def translate_lun2en(text: str) -> str:
    tok_lun2en.src_lang = NLLB_LANG_LUN
    inputs = tok_lun2en(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        out = mdl_lun2en.generate(**inputs, forced_bos_token_id=TGT_EN_ID,
                                   num_beams=5, max_length=256,
                                   no_repeat_ngram_size=3, repetition_penalty=1.3)
    return tok_lun2en.decode(out[0], skip_special_tokens=True).strip()

def _make_translation_response(text: str, direction: str) -> dict:
    nllb = translate_en2lun(text) if direction == "en->lun" else translate_lun2en(text)
    return {"translation": nllb, "translation_nllb": nllb,
            "translation_marian": None, "method": "neural_mt", "confidence": 0.85}

def _qwen_refine(source_en: str, draft_lun: str) -> str:
    if not HF_TOKEN: return draft_lun
    try:
        from openai import OpenAI
        client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=HF_TOKEN)
        resp = client.chat.completions.create(
            model=HF_CHAT_MODEL,
            messages=[
                {"role": "system", "content": "You are a Runyoro-Rutooro grammar expert. Output ONLY the corrected Runyoro text."},
                {"role": "user",   "content": f"English: {source_en}\nDraft: {draft_lun}\nCorrected:"},
            ], max_tokens=128, temperature=0.1)
        first = resp.choices[0].message.content.strip().split("\n")[0].strip()
        if first and "(" not in first and len(first) < 200: return first
    except Exception: pass
    return draft_lun

from transformers import MobileNetV2ForImageClassification, MobileNetV2ImageProcessor
print("Loading MobileNetV2 …")
mn_processor = MobileNetV2ImageProcessor.from_pretrained("google/mobilenet_v2_1.0_224")
mn_model     = MobileNetV2ForImageClassification.from_pretrained("google/mobilenet_v2_1.0_224")
mn_model.eval()
print("  ✓ MobileNetV2 ready")

import easyocr
print("Loading EasyOCR …")
ocr_reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available())
print("  ✓ EasyOCR ready")

print(f"\nSmoke test:")
print(f"  en→lun: 'Good morning' → '{translate_en2lun('Good morning')}'")
print(f"  lun→en: 'Webale muno'  → '{translate_lun2en('Webale muno')}'")

In [ ]:
# ── Cell 6 — FASTAPI APP + SERVER + TUNNEL (all in one) ──────────────────────
# Merged into one cell so ordering issues are impossible.
import base64, json, time, numpy as np, subprocess, threading
from datetime import datetime
from typing import List
from PIL import Image
from fastapi import FastAPI, HTTPException, UploadFile, File, Request
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import requests as _requests

PORT = 7860

# ── App ───────────────────────────────────────────────────────────────────────
app = FastAPI(title="Lunyoro Translator — Colab")
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])

class TranslateReq(BaseModel):
    text: str; context: str = ""; direction: str = "en->lun"; refine: bool = False
class BatchReq(BaseModel):
    sentences: List[str]; direction: str = "en->lun"
class LookupReq(BaseModel):
    word: str; direction: str = "en->lun"
class SpellReq(BaseModel):
    text: str
class FeedbackReq(BaseModel):
    source_text: str; translation: str; direction: str = "en->lun"
    rating: int = 0; correction: str = ""; error_type: str = ""
    model_used: str = ""; domain: str = ""
class ChatReq(BaseModel):
    message: str; history: list = []

_history_store: list = []

@app.get("/")      
def root():         return {"status": "ok", "backend": "colab", "device": device}
@app.get("/health")
def health():       return {"status": "ok", "timestamp": datetime.utcnow().isoformat()}
@app.get("/ping")  
def ping():         return {"pong": True}
@app.get("/system-info")
def system_info():  return {"nllb_en2lun": True, "nllb_lun2en": True,
                             "nllb_en2lun_format": f"fp32_{device}",
                             "nllb_lun2en_format": f"fp32_{device}",
                             "marian_en2lun": False, "marian_lun2en": False,
                             "device": device, "backend": "colab"}
@app.get("/config")
def config():       return {"backend": "colab", "ngrok_url": None, "probe_url": None}

@app.post("/translate")
def translate_text(req: TranslateReq):
    if not req.text.strip(): raise HTTPException(400, "Text cannot be empty")
    r = _make_translation_response(req.text, "en->lun")
    if req.refine and r["translation"]:
        ref = _qwen_refine(req.text, r["translation"])
        r["translation"] = r["translation_nllb"] = ref
    _history_store.insert(0, {"input": req.text, "direction": "en→lun",
                               "translation": r["translation"],
                               "timestamp": datetime.utcnow().isoformat()})
    return r

@app.post("/translate-reverse")
def translate_reverse(req: TranslateReq):
    if not req.text.strip(): raise HTTPException(400, "Text cannot be empty")
    r = _make_translation_response(req.text, "lun->en")
    _history_store.insert(0, {"input": req.text, "direction": "lun→en",
                               "translation": r["translation"],
                               "timestamp": datetime.utcnow().isoformat()})
    return r

@app.post("/translate-batch")
def translate_batch(req: BatchReq):
    if not req.sentences: raise HTTPException(400, "No sentences")
    results = []
    for s in req.sentences[:100]:
        r = _make_translation_response(s, req.direction)
        results.append({"source": s, "translation": r["translation"],
                         "method": r["method"], "confidence": r["confidence"]})
    return {"results": results, "total": len(results), "direction": req.direction}

@app.post("/translate-batch-file")
async def translate_batch_file(file: UploadFile = File(...), direction: str = "en->lun"):
    import csv
    contents = await file.read()
    text = contents.decode("utf-8", errors="ignore")
    sentences = [row[0].strip() for row in csv.reader(text.splitlines()) if row] \
                if (file.filename or "").endswith(".csv") \
                else [l.strip() for l in text.splitlines() if l.strip()]
    results = [{"source": s, "translation": _make_translation_response(s, direction)["translation"],
                "method": "neural_mt"} for s in sentences[:200]]
    return {"results": results, "total": len(results),
            "direction": direction, "filename": file.filename}

@app.post("/classify-image")
async def classify_image(file: UploadFile = File(...), top_k: int = 1):
    contents = await file.read()
    try:    img = Image.open(io.BytesIO(contents)).convert("RGB")
    except Exception as e: raise HTTPException(400, f"Cannot decode image: {e}")
    inputs = mn_processor(images=img, return_tensors="pt")
    with torch.no_grad(): logits = mn_model(**inputs).logits
    probs = torch.nn.functional.softmax(logits, dim=-1)[0]
    top_probs, top_idx = torch.topk(probs, min(top_k, 5))
    results = []
    for prob, idx in zip(top_probs, top_idx):
        label_en  = mn_model.config.id2label[idx.item()].split(",")[0].strip().lower()
        label_lun = translate_en2lun(label_en)
        results.append({"label_en": label_en, "label_lun": _qwen_refine(label_en, label_lun),
                         "label_lun_raw": label_lun, "confidence": round(prob.item(), 4),
                         "method": "nllb+qwen" if HF_TOKEN else "nllb"})
    return {"predictions": results, "top_k": len(results), "model": "google/mobilenet_v2_1.0_224"}

@app.get("/classify-image/status")
def classify_status(): return {"ready": True, "error": None}

def _ocr_and_translate(img_bytes: bytes, direction: str) -> dict:
    import cv2
    arr = np.frombuffer(img_bytes, np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if img is None: return {"error": "Cannot decode image", "regions": []}
    h, w = img.shape[:2]
    regions = []
    for (bbox, text, conf) in ocr_reader.readtext(img):
        if conf < 0.3 or not text.strip(): continue
        translated = _make_translation_response(text, direction)["translation"]
        xs = [p[0] for p in bbox]; ys = [p[1] for p in bbox]
        x1,y1,x2,y2 = min(xs),min(ys),max(xs),max(ys)
        regions.append({"original": text, "translated": translated,
                         "confidence": round(conf, 2),
                         "bbox": {"x":int(x1),"y":int(y1),"width":int(x2-x1),"height":int(y2-y1)},
                         "bbox_norm": {"x":round(x1/w,4),"y":round(y1/h,4),
                                       "width":round((x2-x1)/w,4),"height":round((y2-y1)/h,4)}})
    return {"regions": regions, "total_detected": len(regions),
            "total_translated": len(regions),
            "image_size": {"width": w, "height": h}, "direction": direction}

@app.post("/ocr-translate")
async def ocr_translate(file: UploadFile = File(...), direction: str = "en->lun"):
    return _ocr_and_translate(await file.read(), direction)

@app.post("/ocr-translate-base64")
async def ocr_translate_b64(request: Request):
    body = await request.json()
    img_data = body.get("image", "")
    if "," in img_data: img_data = img_data.split(",", 1)[1]
    try:    img_bytes = base64.b64decode(img_data)
    except Exception as e: return {"error": f"Bad base64: {e}", "regions": []}
    return _ocr_and_translate(img_bytes, body.get("direction", "en->lun"))

@app.post("/chat")
async def chat(req: ChatReq):
    SYSTEM = ("You are a helpful assistant for the Runyoro-Rutooro language of Uganda. "
              "Reply in clear, simple English (2-4 sentences). "
              "Your reply will be translated into Runyoro-Rutooro — do NOT include Runyoro words.")
    english = ""
    if HF_TOKEN:
        try:
            from openai import OpenAI
            client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=HF_TOKEN)
            msgs = [{"role": "system", "content": SYSTEM}]
            for h in req.history[-4:]:
                msgs.append({"role": h.get("role","user"), "content": h.get("content","")})
            msgs.append({"role": "user", "content": req.message})
            resp = client.chat.completions.create(model=HF_CHAT_MODEL, messages=msgs,
                                                   max_tokens=256, temperature=0.7)
            english = resp.choices[0].message.content.strip()
        except Exception: pass
    if not english:
        english = "I can help you learn Runyoro-Rutooro. Try asking me to translate a phrase."
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", english) if s.strip()]
    runyoro = " ".join(translate_en2lun(s) for s in sentences)
    return {"reply": f"{runyoro}\n\n(English: {english})",
            "reply_nllb": runyoro, "reply_marian": None}

@app.post("/lookup")
def lookup(req: LookupReq):
    direction = "en->lun" if "en" in req.direction.lower() else "lun->en"
    r = _make_translation_response(req.word, direction)
    return {"word": req.word, "results": [{"translation": r["translation"], "method": "nllb"}]}

@app.post("/spellcheck")
def spellcheck(req: SpellReq): return {"misspelled": []}

@app.get("/language-rules/interjections")
def interjections(): return {"interjections": {
    "Agandi": "How are you?", "Oraire ota": "Good morning",
    "Webale muno": "Thank you very much", "Mpora": "Take it easy",
    "Genda kurungi": "Goodbye / Go well"}}

@app.get("/language-rules/idioms")
def idioms(): return {"idioms": {}}

@app.post("/feedback")
def feedback(req: FeedbackReq): return {"status": "saved"}

@app.get("/history")
def history(): return {"history": _history_store[:100]}

@app.post("/summarize-pdf")
async def summarize_pdf(file: UploadFile = File(...)):
    raise HTTPException(501, "PDF summarization not available in Colab backend.")

print("✓ FastAPI app defined — starting server...")

# ── Start uvicorn in background thread ───────────────────────────────────────
def _run_uvicorn():
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

threading.Thread(target=_run_uvicorn, daemon=True).start()
time.sleep(4)

try:
    r = _requests.get(f"http://localhost:{PORT}/health", timeout=5)
    assert r.status_code == 200
    print(f"✓ Server running on port {PORT}")
except Exception as e:
    print(f"✗ Server not responding: {e}")
    raise

# ── Install + start cloudflared tunnel ───────────────────────────────────────
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("Installing cloudflared …")
    subprocess.run(["wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "/usr/local/bin/cloudflared"], check=True)
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url = None
print("Waiting for tunnel", end="", flush=True)
for _ in range(60):
    line = cf_proc.stdout.readline()
    m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        break
    print(".", end="", flush=True)
print()

if not public_url:
    raise RuntimeError("Could not get cloudflared URL")
print(f"✓ Tunnel: {public_url}")

# ── Register with Cloudflare Worker ──────────────────────────────────────────
try:
    headers = {"Content-Type": "application/json"}
    if WORKER_SECRET: headers["X-Worker-Secret"] = WORKER_SECRET
    resp = _requests.post(f"{WORKER_URL}/update-colab-url",
                          json={"url": public_url}, headers=headers, timeout=15)
    if resp.status_code == 200:
        print("✓ Worker updated — Vercel now routes to this Colab T4")
    else:
        print(f"⚠ Worker returned {resp.status_code}: {resp.text[:80]}")
except Exception as e:
    print(f"⚠ Could not reach Worker: {e}")

print()
print("=" * 60)
print(f"  Tunnel  : {public_url}")
print(f"  Worker  : {WORKER_URL}")
print(f"  GPU     : {device}")
print(f"  Frontend: https://horizonx.kathay.tech")
print("=" * 60)
print("Keep this notebook running. Tunnel closes when runtime stops.")

In [ ]:
# ── Cell 7 — VERIFY END-TO-END ────────────────────────────────────────────────
import requests
BASE = f"http://localhost:{PORT}"
h  = requests.get(f"{BASE}/health", timeout=5).json()
t1 = requests.post(f"{BASE}/translate",
     json={"text": "How are you?", "direction": "en->lun"}, timeout=30).json()
t2 = requests.post(f"{BASE}/translate-reverse",
     json={"text": "Webale muno", "direction": "lun->en"}, timeout=30).json()
print("Health :", h)
print("en→lun : 'How are you?'  →", t1["translation"])
print("lun→en : 'Webale muno'  →", t2["translation"])
print("\n✓ All checks passed — horizonx.kathay.tech is live on Colab T4")